# 2.15 Black-box & derivative-free optimization

Derivative-free methods buy progress using function comparisons when gradients are missing or untrustworthy.

Gradient descent shows what a direct local direction can do. Here the optimizer uses finite differences, direct comparisons, and a real Nelder-Mead simplex so it can operate when gradients are unavailable or unreliable.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 215
rng = np.random.default_rng(SEED)


def finite_difference_gradient(function, x, h=1e-3):
    x = np.asarray(x, dtype=float)
    gradient = np.zeros_like(x)

    for index in range(len(x)):
        step = np.zeros_like(x)
        step[index] = h
        gradient[index] = (function(x + step) - function(x - step)) / (2.0 * h)

    return gradient


def pattern_search(function, start, step_size=1.0, shrink=0.5, iterations=20):
    x = np.asarray(start, dtype=float).copy()
    best_value = float(function(x))
    values = [best_value]
    path = [x.copy()]

    for iteration in range(iterations):
        improved = False
        directions = np.eye(len(x))
        directions = np.vstack([directions, -directions])

        for direction in directions:
            candidate = x + step_size * direction
            candidate_value = float(function(candidate))
            if candidate_value < best_value:
                x = candidate
                best_value = candidate_value
                improved = True
                path.append(x.copy())
                values.append(best_value)
                break

        if not improved:
            step_size = step_size * shrink
            values.append(best_value)
            path.append(x.copy())

    return np.asarray(path), np.asarray(values)


def nelder_mead(function, start, step=0.25, iterations=80, alpha=1.0, gamma=2.0, rho=0.5, sigma=0.5):
    start = np.asarray(start, dtype=float)
    dimension = len(start)
    simplex = [start]

    for index in range(dimension):
        vertex = start.copy()
        vertex[index] = vertex[index] + step
        simplex.append(vertex)

    simplex = np.asarray(simplex, dtype=float)
    values = np.array([function(vertex) for vertex in simplex], dtype=float)
    best_curve = [float(np.min(values))]
    path = [simplex[int(np.argmin(values))].copy()]

    for iteration in range(iterations):
        order = np.argsort(values)
        simplex = simplex[order]
        values = values[order]
        centroid = np.mean(simplex[:-1], axis=0)
        worst = simplex[-1]
        reflected = centroid + alpha * (centroid - worst)
        reflected_value = float(function(reflected))

        if values[0] <= reflected_value < values[-2]:
            simplex[-1] = reflected
            values[-1] = reflected_value
        elif reflected_value < values[0]:
            expanded = centroid + gamma * (reflected - centroid)
            expanded_value = float(function(expanded))
            if expanded_value < reflected_value:
                simplex[-1] = expanded
                values[-1] = expanded_value
            else:
                simplex[-1] = reflected
                values[-1] = reflected_value
        else:
            contracted = centroid + rho * (worst - centroid)
            contracted_value = float(function(contracted))
            if contracted_value < values[-1]:
                simplex[-1] = contracted
                values[-1] = contracted_value
            else:
                best = simplex[0].copy()
                for index in range(1, len(simplex)):
                    simplex[index] = best + sigma * (simplex[index] - best)
                    values[index] = float(function(simplex[index]))

        best_curve.append(float(np.min(values)))
        path.append(simplex[int(np.argmin(values))].copy())

    return {
        "best_x": simplex[int(np.argmin(values))].copy(),
        "best_value": float(np.min(values)),
        "best_curve": np.asarray(best_curve),
        "path": np.asarray(path),
    }


def repeated_objective(function, repeats=5, noise_scale=0.0):
    def wrapped(x):
        values = []
        for repeat in range(repeats):
            noise = rng.normal(0.0, noise_scale)
            values.append(function(x) + noise)
        return float(np.mean(values))

    return wrapped


def sigmoid(scores):
    return 1.0 / (1.0 + np.exp(-np.clip(scores, -30, 30)))


def load_logistic_data(max_features=8):
    try:
        from sklearn.datasets import load_breast_cancer

        data = load_breast_cancer()
        features = data.data[:, :max_features].astype(float)
        target = data.target.astype(float)
    except Exception:
        features = rng.normal(size=(180, max_features))
        true_weights = np.linspace(-1.0, 1.0, max_features)
        target = (sigmoid(features @ true_weights) > 0.5).astype(float)

    split = int(0.7 * len(features))
    train_x = features[:split]
    valid_x = features[split:]
    train_y = target[:split]
    valid_y = target[split:]
    mean = train_x.mean(axis=0)
    scale = train_x.std(axis=0) + 1e-6
    train_x = (train_x - mean) / scale
    valid_x = (valid_x - mean) / scale
    return train_x, train_y, valid_x, valid_y


LOGISTIC_DATA = load_logistic_data(max_features=8)


def logistic_black_box_loss(point):
    log_lr, threshold = point
    learning_rate = 10.0 ** log_lr
    l2 = 1e-3
    train_x, train_y, valid_x, valid_y = LOGISTIC_DATA
    weights = np.zeros(train_x.shape[1])
    bias = 0.0

    for step in range(80):
        prediction = sigmoid(train_x @ weights + bias)
        error = prediction - train_y
        gradient = train_x.T @ error / len(train_y) + l2 * weights
        bias_gradient = float(np.mean(error))
        weights = weights - learning_rate * gradient
        bias = bias - learning_rate * bias_gradient

    probabilities = sigmoid(valid_x @ weights + bias)
    predictions = (probabilities >= threshold).astype(float)
    error_rate = np.mean(predictions != valid_y)
    calibration = np.mean((probabilities - valid_y) ** 2)
    return float(error_rate + 0.1 * calibration)


def build_derivative_free_ladder():
    def d1(point):
        x, y = point
        return (x - 1.0) ** 2 + (y + 2.0) ** 2

    def d2(point):
        x, y = point
        return 0.02 * (x - 3.0) ** 2 + 8.0 * (y + 0.5) ** 2

    def d3(point):
        x, y = point
        rosenbrock = (1.0 - x) ** 2 + 20.0 * (y - x * x) ** 2
        waves = 0.05 * np.sin(6 * x) * np.cos(6 * y)
        return rosenbrock + waves

    def d5(point):
        x = np.asarray(point)
        hinge = np.maximum(0.0, np.sum(np.abs(x)) - 2.0)
        rugged = 0.04 * np.sum(np.sin(5 * x) ** 2)
        return float(np.sum((x - np.array([0.4, -0.7, 0.2, 0.9])) ** 2) + rugged + 2.0 * hinge)

    return [
        {"name": "D1 quadratic bowl", "objective": d1, "start": np.array([0.0, 0.0]), "step": 1.0},
        {"name": "D2 ill-conditioned bowl", "objective": d2, "start": np.array([-2.0, 1.5]), "step": 1.0},
        {"name": "D3 rugged Rosenbrock", "objective": d3, "start": np.array([-1.2, 1.0]), "step": 0.5},
        {"name": "D4 real logistic validation black box", "objective": logistic_black_box_loss, "start": np.array([-1.0, 0.5]), "step": 0.3},
        {"name": "D5 constrained nondifferentiable surface", "objective": d5, "start": np.array([1.4, 1.2, -1.0, -1.1]), "step": 0.6},
    ]


## The concept, built once: finite differences and direct search

A central finite-difference slope is $\frac{f(x+h)-f(x-h)}{2h}$. Direct search accepts points by comparing objective values instead of using an analytic gradient.

In [ ]:

def lesson_one_dimensional(point):
    x = point[0]
    return (x - 2.0) ** 2

f_plus = lesson_one_dimensional(np.array([0.1]))
f_minus = lesson_one_dimensional(np.array([-0.1]))
slope = finite_difference_gradient(lesson_one_dimensional, np.array([0.0]), h=0.1)[0]

print("f(0.1)", f_plus)
print("f(-0.1)", f_minus)
print("central slope", slope)

assert abs(f_plus - 3.61) < 1e-12
assert abs(f_minus - 4.41) < 1e-12
assert abs(slope + 4.00) < 1e-12


## Pattern search matches the lesson path

For $f(x,y)=(x-1)^2+(y+2)^2$, the lesson moves from value $5$ to $4$ to $1$ to $0$ using only comparisons.

In [ ]:

def lesson_two_dimensional(point):
    x, y = point
    return (x - 1.0) ** 2 + (y + 2.0) ** 2

manual_points = [
    np.array([0.0, 0.0]),
    np.array([1.0, 0.0]),
    np.array([1.0, -1.0]),
    np.array([1.0, -2.0]),
]
manual_values = [lesson_two_dimensional(point) for point in manual_points]
pattern_path, pattern_values = pattern_search(lesson_two_dimensional, np.array([0.0, 0.0]), step_size=1.0, iterations=3)

print("manual values", manual_values)
print("pattern-search values", pattern_values.tolist())

assert manual_values == [5.0, 4.0, 1.0, 0.0]
assert pattern_values[-1] == 0.0


## The dataset ladder

D1 is a simple bowl, D2 is ill-conditioned, D3 is a rugged Rosenbrock variant, D4 mimics validation-metric tuning, and D5 is high-dimensional, constrained, and nondifferentiable.

In [ ]:

rungs = build_derivative_free_ladder()

for index, rung in enumerate(rungs, start=1):
    start = rung["start"]
    value = rung["objective"](start)
    print(f"D{index}: {rung['name']}")
    print("  dimension", len(start), "start", np.round(start, 3).tolist())
    print("  start loss", round(value, 4))


## Run the same method across D1-D5

Every rung uses the same Nelder-Mead simplex optimizer. The metric is best observed loss; lower is better.

In [ ]:

derivative_free_results = []

for index, rung in enumerate(rungs, start=1):
    result = nelder_mead(
        rung["objective"],
        rung["start"],
        step=rung["step"],
        iterations=70,
    )
    derivative_free_results.append(result)
    print(f"D{index} | best observed loss {result['best_value']:.5f} | iterations {len(result['best_curve'])}")


## Results visualization

The left panels show Nelder-Mead best-path coordinates. The right panel shows best observed loss versus iteration.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
flat_axes = axes.ravel()

for index, result in enumerate(derivative_free_results):
    ax = flat_axes[index]
    path = result["path"]
    if path.shape[1] == 1:
        ax.plot(path[:, 0], marker="o")
        ax.set_xlabel("iteration")
        ax.set_ylabel("x")
    else:
        ax.plot(path[:, 0], path[:, 1], marker="o", linewidth=1)
        ax.set_xlabel("x0")
        ax.set_ylabel("x1")
    ax.set_title(f"D{index + 1} simplex path")

summary_ax = flat_axes[-1]
for index, result in enumerate(derivative_free_results, start=1):
    summary_ax.plot(result["best_curve"], label=f"D{index}")

summary_ax.set_title("best observed loss vs iteration")
summary_ax.set_xlabel("Nelder-Mead iteration")
summary_ax.set_ylabel("best loss")
summary_ax.legend()
plt.tight_layout()


## Pitfall on D5: one noisy comparison

A single noisy evaluation can move the search toward luck rather than signal. The fix is to average repeated evaluations before comparisons; finite differences also need scale-aware $h$ instead of a one-size-fits-all step.

In [ ]:

d5 = rungs[-1]
noisy_once = repeated_objective(d5["objective"], repeats=1, noise_scale=0.25)
noisy_average = repeated_objective(d5["objective"], repeats=7, noise_scale=0.25)

one_shot = nelder_mead(noisy_once, d5["start"], step=d5["step"], iterations=50)
averaged = nelder_mead(noisy_average, d5["start"], step=d5["step"], iterations=50)
true_one_shot = d5["objective"](one_shot["best_x"])
true_averaged = d5["objective"](averaged["best_x"])

print("true loss after one noisy comparison", true_one_shot)
print("true loss after averaged comparisons", true_averaged)

assert np.isfinite(true_one_shot)
assert np.isfinite(true_averaged)


## Evaluate it + Practice

- Metric: track best observed loss against a no-skill baseline: random perturbations around the same start point.
- Cheap sanity check: finite difference at x=0 for (x-2)^2 must be -4.00 with h=0.1.
- Ablation: turn off reflection/expansion by shrinking the simplex every step and compare curves.
- Failure signals: simplex collapses early, noisy wins do not reproduce, or dimension makes comparisons explode.

### Practice

- Change the starting simplex size on D2 and explain under- versus over-shooting.

- Estimate a central-difference gradient on D5 with two values of h and compare stability.

- Add repeated evaluations to D4 and measure the cost-quality tradeoff.